# Phase 3: Text Vectorization & Content-Based Recommendation Engine

**Project**: Personalized Movie Recommendation System  
**Repository**: `Dp8453/Personalized-movie-recommendation-system`  
**Input Dataset**: `data/processed/clean_movies.csv` (4,803 movies)  

---  
### Objectives of Phase 3:
1. Load the processed dataset containing Phase 2 `tags` metadata.
2. Transform textual `tags` into numerical feature vectors using `TfidfVectorizer(max_features=5000, stop_words='english')`.
3. Compute pairwise Cosine Similarity matrix across all 4,803 movie feature vectors.
4. Implement `MovieRecommender` class supporting robust case-insensitive title lookups.
5. Generate and evaluate top-N content-based recommendations for sample query movies.
6. Perform data quality and ranking verification checks.

## 1. Import Required Libraries & Modules

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure src module can be imported from parent directory
sys.path.append(os.path.abspath('..'))
from src.recommender import MovieRecommender

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Imports successful.')

## 2. Load Processed Dataset (`data/processed/clean_movies.csv`)

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', 'clean_movies.csv')
df = pd.read_csv(processed_path)
print(f'Processed Dataset Shape: {df.shape}')
print(f"Total Movies: {len(df)}")
print(f"Columns: {list(df.columns)}\n")
df[['id', 'title', 'tags']].head(3)

### 2.1 Verify Dataset Integrity
- Exactly **4,803 movie records**.
- `tags` column exists and contains non-empty strings.

In [ ]:
assert len(df) == 4803, f'Expected 4,803 movies, got {len(df)}'
assert 'tags' in df.columns, "'tags' column missing"
assert df['tags'].isnull().sum() == 0, 'Null tags found'
print('Dataset integrity checks passed.')

## 3. TF-IDF Text Vectorization Explanation & Setup

**TF-IDF (Term Frequency-Inverse Document Frequency)** converts text documents into numerical vectors by weighing word occurrences:
- **TF (Term Frequency)**: Measures how often a word appears in a specific movie's `tags`.
- **IDF (Inverse Document Frequency)**: Measures how rare a word is across all 4,803 movies.
- **TF-IDF Weight**: $\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$. Words appearing in almost all movies (like "movie" or "the") receive low weight, whereas specific keywords (like "gotham", "jamescameron", "spacecolony") receive high weight.

### Vectorizer Configuration:
- `max_features = 5000` (retains top 5,000 most informative vocabulary terms)
- `stop_words = 'english'` (removes common English stop words)

## 4. Cosine Similarity Modeling Explanation

**Cosine Similarity** measures the cosine of the angle between two multi-dimensional TF-IDF vectors $A$ and $B$:
$$\text{Similarity}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$$
- Value ranges from **0.0** (completely orthogonal / no overlapping words) to **1.0** (identical content representations).
- For 4,803 movies, the resulting Cosine Similarity matrix is of shape **(4803, 4803)**.

## 5. Instantiate `MovieRecommender` Engine

In [ ]:
recommender = MovieRecommender(df, max_features=5000, stop_words='english')

print(f"TF-IDF Matrix Dimensions : {recommender.get_tfidf_matrix_shape()}")
print(f"Vocabulary Size          : {recommender.get_vocab_size()}")
print(f"Cosine Similarity Shape  : {recommender.get_similarity_matrix_shape()}")

## 6. Example Movie Recommendations

In [ ]:
test_movies = ['Avatar', 'The Dark Knight', 'Inception']

for movie_title in test_movies:
    print(f"\n========================================")
    print(f"Input Query Movie: '{movie_title}'")
    print(f"========================================")
    recs = recommender.recommend(movie_title, top_n=5)
    for idx, r in enumerate(recs, 1):
        print(f"  {idx}. {r['title']:35s} | Similarity Score: {r['similarity_score']:.4f}")

## 7. Case-Insensitive Title Lookup Test

In [ ]:
query_variants = ['Avatar', 'avatar', 'AVATAR', '  Avatar  ']
print('Testing case-insensitive title lookups:')

results = {}
for variant in query_variants:
    recs = recommender.recommend(variant, top_n=3)
    results[variant] = recs
    print(f"Query '{variant}': Top recommendation = '{recs[0]['title']}' (Score: {recs[0]['similarity_score']})")

# Assert all variants yield identical recommendations
first_res = list(results.values())[0]
for res in results.values():
    assert res == first_res, 'Case-insensitive lookup returned inconsistent results!'
print('\nCase-insensitive title lookup verification passed.')

## 8. Invalid Title Error Handling Test

In [ ]:
invalid_title = 'NonExistentMovieMovie999'
try:
    recommender.recommend(invalid_title)
    print('FAIL: Expected ValueError was not raised!')
except ValueError as e:
    print(f"SUCCESS: Caught expected ValueError for '{invalid_title}':\n  -> {e}")

## 9. Recommendation Quality & Ranking Verification

In [ ]:
query = 'Avatar'
top_5 = recommender.recommend(query, top_n=5)

# Check 1: Query movie self-exclusion
rec_titles = [r['title'] for r in top_5]
assert query not in rec_titles, f'Query movie {query} found in recommendations!'

# Check 2: No duplicate recommendations
assert len(rec_titles) == len(set(rec_titles)), 'Duplicate recommendation titles found!'

# Check 3: Descending sort order
scores = [r['similarity_score'] for r in top_5]
assert scores == sorted(scores, reverse=True), 'Recommendations not sorted descending!'

# Check 4: Score boundaries [0.0, 1.0]
assert all(0.0 <= s <= 1.0 for s in scores), 'Scores outside valid [0, 1] range!'

# Check 5: Top N count
assert len(top_5) == 5, f'Expected 5 recommendations, got {len(top_5)}'

print('All 5 recommendation quality assertions passed cleanly.')

## 10. Phase 3 Summary & Conclusions

### Complete Pipeline Architecture:
```
  [ Raw Movies & Credits CSVs ]
               │
               ▼  (Phase 1: ID-Based Primary Key Merging)
  [ Merged Dataset (4,803 Movies) ]
               │
               ▼  (Phase 2: JSON Extraction, Space Collapsing & Overview Imputation)
  [ Cleaned Feature Dataset & Unified Tags ]
               │
               ▼  (Phase 3: TF-IDF Vectorization)
  [ TF-IDF Feature Matrix (4803 x 5000) ]
               │
               ▼  (Phase 3: Cosine Similarity Matrix)
  [ Pairwise Cosine Similarity Matrix (4803 x 4803) ]
               │
               ▼  (Phase 3: Case-Insensitive Lookup & Score Ranking)
  [ Top-N Ranked Content-Based Movie Recommendations ]
```

- **Phase 3 Objective Achieved**: Built a classical Content-Based Movie Recommendation Engine.
- **Next Phase**: Phase 4 will introduce User Personalization & Hybrid Ranking Logic.